In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/DVA_Project')

RAW_PATH = PROJECT_ROOT / 'data/raw/retail_store_sales.csv'
INTERIM_PATH = PROJECT_ROOT / 'data/interim/extracted.csv'

INTERIM_PATH.parent.mkdir(parents=True, exist_ok=True)

In [4]:
import pandas as pd

df = pd.read_csv(RAW_PATH)
df.head()

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [5]:
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])
df.info()

Rows: 12575
Columns: 11
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  object 
 1   Customer ID       12575 non-null  object 
 2   Category          12575 non-null  object 
 3   Item              11362 non-null  object 
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  object 
 8   Location          12575 non-null  object 
 9   Transaction Date  12575 non-null  object 
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(8)
memory usage: 1.1+ MB


In [6]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r'\s+', '_', regex=True)
)

In [7]:
df = df.drop_duplicates()

In [8]:
print(df.columns.tolist())

['transaction_id', 'customer_id', 'category', 'item', 'price_per_unit', 'quantity', 'total_spent', 'payment_method', 'location', 'transaction_date', 'discount_applied']


In [13]:
# ---- SAFE TYPE CONVERSION ----

numeric_cols = ['price_per_unit', 'quantity', 'total_spent']

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Handle transaction_date ONLY if not already datetime
if 'transaction_date' in df.columns:
    if not pd.api.types.is_datetime64_any_dtype(df['transaction_date']):
        df['transaction_date'] = pd.to_datetime(
            df['transaction_date'].astype(str).str.strip(),
            dayfirst=True,
            errors='coerce'
        )
    else:
        print("transaction_date already in datetime format → skipping conversion")

# Validation
print(df.dtypes)
print("\nMissing:")
print(df[['transaction_date']].isnull().sum())

transaction_date already in datetime format → skipping conversion
transaction_id              object
customer_id                 object
category                    object
item                        object
price_per_unit             float64
quantity                   float64
total_spent                float64
payment_method              object
location                    object
transaction_date    datetime64[ns]
discount_applied            object
dtype: object

Missing:
transaction_date    0
dtype: int64


In [15]:
df['discount_applied'] = (
    df['discount_applied']
    .astype(str)
    .str.strip()
    .str.upper()
    .map({'TRUE': True, 'FALSE': False})
)

In [16]:
print(df['discount_applied'].value_counts(dropna=False))
print(df['discount_applied'].dtype)

discount_applied
True     4219
NaN      4199
False    4157
Name: count, dtype: int64
object


In [17]:
print(df.dtypes)
print("\nMissing %:\n", (df.isnull().sum() / len(df)) * 100)
print("\nShape:", df.shape)

transaction_id              object
customer_id                 object
category                    object
item                        object
price_per_unit             float64
quantity                   float64
total_spent                float64
payment_method              object
location                    object
transaction_date    datetime64[ns]
discount_applied            object
dtype: object

Missing %:
 transaction_id       0.000000
customer_id          0.000000
category             0.000000
item                 9.646123
price_per_unit       4.842942
quantity             4.803181
total_spent          4.803181
payment_method       0.000000
location             0.000000
transaction_date     0.000000
discount_applied    33.391650
dtype: float64

Shape: (12575, 11)


In [18]:
df.to_csv('/content/drive/MyDrive/DVA_Project/data/interim/extracted.csv', index=False)